In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, requests, pandas as pd
from dotenv import load_dotenv

load_dotenv("/Users/samarkumar/Documents/GitHub/AerisAI/aerisai/.env")
API_KEY = os.getenv("OPENAQ_API_KEY")
assert API_KEY, "Key not found - check the .env file name and location"

headers = {"X-API-Key": API_KEY}
url = "https://api.openaq.org/v3/locations"
params = {
    "coordinates": "28.57,77.33",   # approx. centre of Noida (lat,lon)
    "radius": 25000,                # metres (25 km is the maximum)
    "limit": 100,
}

r = requests.get(url, headers=headers, params=params)
r.raise_for_status()
results = r.json()["results"]
print(len(results), "locations found")

rows = []
for loc in results:
    measured = sorted({s["parameter"]["name"] for s in loc.get("sensors", [])})
    rows.append({
        "id": loc["id"],
        "name": loc["name"],
        "lat": loc["coordinates"]["latitude"],
        "lon": loc["coordinates"]["longitude"],
        "provider": (loc.get("provider") or {}).get("name"),
        "first": (loc.get("datetimeFirst") or {}).get("utc"),
        "last": (loc.get("datetimeLast") or {}).get("utc"),
        "measures": ", ".join(measured),
    })

stations = pd.DataFrame(rows).sort_values("last", ascending=False)
stations

93 locations found


,id,name,lat,lon,provider,first,last,measures
58,8118,New Delhi,28.635760,77.224450,AirNow,2016-11-09T19:00:00Z,2026-10-10T13:30:00Z,pm25
76,2860223,GK1 (Oberoi Terrace),28.550005,77.243106,AirGradient,2024-05-22T11:00:00Z,2026-10-10T13:00:00Z,"pm1, pm10, pm25, relativehumidity, temperature..."
78,4663956,"Santushti Apartments, Vasant Kunj",28.526600,77.149300,AirGradient,2025-06-08T09:00:00Z,2026-10-10T13:00:00Z,"pm1, pm25, relativehumidity, temperature, um003"
79,4712609,Air Check,28.623056,77.234167,AirGradient,2026-01-28T11:00:00Z,2026-10-10T13:00:00Z,"pm1, pm25, relativehumidity, temperature, um003"
80,6105800,"Sector 1, Noida extension",28.583491,77.431714,AirGradient,2025-10-23T18:00:00Z,2026-10-10T13:00:00Z,"pm1, pm25, relativehumidity, temperature, um003"
...,...,...,...,...,...,...,...,...
8,2503,"Shadipur, Delhi - CPCB",28.651700,77.158100,CPCB,2016-03-21T23:30:00Z,2018-02-22T02:45:00Z,"co, no2, o3, pm25, so2"
6,236,"Mandir Marg, Delhi - DPCC",28.634100,77.200500,CPCB,2016-02-05T15:00:00Z,2018-02-21T21:15:00Z,"co, no2, o3, pm10, pm25, so2"
10,2597,US Diplomatic Post: New Delhi,28.598096,77.189066,StateAir NewDelhi,2016-01-30T00:30:00Z,2016-11-09T16:30:00Z,pm25
0,15,IGI Airport,28.560000,77.094000,CPCB,NaN,NaN,"co, no2, o3, pm10, pm25"


In [5]:
pm25 = stations[stations["measures"].str.contains("pm25")]
print(len(pm25), "stations measure PM2.5")
pm25.to_csv("../aerisai/data/raw/station_candidates.csv", index=False)
pm25

91 stations measure PM2.5


,id,name,lat,lon,provider,first,last,measures
58,8118,New Delhi,28.635760,77.224450,AirNow,2016-11-09T19:00:00Z,2026-10-10T13:30:00Z,pm25
76,2860223,GK1 (Oberoi Terrace),28.550005,77.243106,AirGradient,2024-05-22T11:00:00Z,2026-10-10T13:00:00Z,"pm1, pm10, pm25, relativehumidity, temperature..."
78,4663956,"Santushti Apartments, Vasant Kunj",28.526600,77.149300,AirGradient,2025-06-08T09:00:00Z,2026-10-10T13:00:00Z,"pm1, pm25, relativehumidity, temperature, um003"
79,4712609,Air Check,28.623056,77.234167,AirGradient,2026-01-28T11:00:00Z,2026-10-10T13:00:00Z,"pm1, pm25, relativehumidity, temperature, um003"
80,6105800,"Sector 1, Noida extension",28.583491,77.431714,AirGradient,2025-10-23T18:00:00Z,2026-10-10T13:00:00Z,"pm1, pm25, relativehumidity, temperature, um003"
...,...,...,...,...,...,...,...,...
8,2503,"Shadipur, Delhi - CPCB",28.651700,77.158100,CPCB,2016-03-21T23:30:00Z,2018-02-22T02:45:00Z,"co, no2, o3, pm25, so2"
6,236,"Mandir Marg, Delhi - DPCC",28.634100,77.200500,CPCB,2016-02-05T15:00:00Z,2018-02-21T21:15:00Z,"co, no2, o3, pm10, pm25, so2"
10,2597,US Diplomatic Post: New Delhi,28.598096,77.189066,StateAir NewDelhi,2016-01-30T00:30:00Z,2016-11-09T16:30:00Z,pm25
0,15,IGI Airport,28.560000,77.094000,CPCB,NaN,NaN,"co, no2, o3, pm10, pm25"


In [6]:
import os, time, json, requests, pandas as pd
from dotenv import load_dotenv

RAW = "../aerisai/data/raw"
load_dotenv("../aerisai/.env")      # use whichever .env path worked before
headers = {"X-API-Key": os.getenv("OPENAQ_API_KEY")}

stations = pd.read_csv(f"{RAW}/station_candidates.csv")
shortlist_ids = [5616, 5598, 6988, 6980, 6978, 6986, 5665, 6924, 10919, 7005, 235, 6359]

short = stations[stations["id"].isin(shortlist_ids)].sort_values("name")
print(len(short), "stations matched")
short[["id", "name", "lat", "lon", "first", "last"]]

12 stations matched


,id,name,lat,lon,first,last
8,235,"Anand Vihar, New Delhi - DPCC",28.646835,77.316032,2016-02-05T14:15:00Z,2026-10-07T14:30:00Z
24,6359,"IHBAS, Dilshad Garden,New Delhi - CPCB",28.681174,77.302523,2018-04-28T15:45:00Z,2026-10-07T14:30:00Z
25,6924,"Indirapuram, Ghaziabad - UPPCB",28.646233,77.358075,2019-04-27T12:00:00Z,2026-10-07T14:30:00Z
30,6978,"Knowledge Park - III, Greater Noida - UPPCB",28.472720,77.482000,2018-08-10T06:00:00Z,2026-10-07T14:30:00Z
32,6986,"Knowledge Park - V, Greater Noida - UPPCB",28.557054,77.453663,2019-06-28T06:00:00Z,2026-10-07T14:30:00Z
34,7005,"Loni, Ghaziabad - UPPCB",28.757294,77.278792,2019-04-27T11:45:00Z,2026-10-07T14:30:00Z
42,10919,"Sanjay Nagar, Ghaziabad - UPPCB",28.685382,77.453839,2019-04-27T12:00:00Z,2026-10-07T14:30:00Z
13,5598,"Sector - 125, Noida, UP - UPPCB",28.544761,77.323126,2025-02-18T20:15:00Z,2026-10-07T14:30:00Z
16,5616,"Sector - 62, Noida, UP - IMD",28.624548,77.357710,2025-02-18T20:13:00Z,2026-10-07T14:30:00Z
31,6980,"Sector-1, Noida - UPPCB",28.589247,77.321962,2019-06-28T05:45:00Z,2026-10-07T14:30:00Z


In [7]:
sensor_rows = []
for loc_id in short["id"]:
    r = requests.get(f"https://api.openaq.org/v3/locations/{loc_id}", headers=headers)
    r.raise_for_status()
    loc = r.json()["results"][0]
    for s in loc["sensors"]:
        sensor_rows.append({
            "location_id": loc_id,
            "location": loc["name"],
            "sensor_id": s["id"],
            "parameter": s["parameter"]["name"],
            "units": s["parameter"]["units"],
        })
    time.sleep(1.1)     # stay under the 60 requests/minute limit

sensors = pd.DataFrame(sensor_rows)
sensors = sensors[sensors["parameter"].isin(["pm25", "pm10"])]
sensors.to_csv(f"{RAW}/sensor_list.csv", index=False)
sensors

,location_id,location,sensor_id,parameter,units
8,235,"Anand Vihar, New Delhi - DPCC",12235609,pm10,µg/m³
9,235,"Anand Vihar, New Delhi - DPCC",381,pm10,µg/m³
10,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,µg/m³
11,235,"Anand Vihar, New Delhi - DPCC",384,pm25,µg/m³
26,6359,"IHBAS, Dilshad Garden,New Delhi - CPCB",36050,pm10,µg/m³
27,6359,"IHBAS, Dilshad Garden,New Delhi - CPCB",12234734,pm10,µg/m³
28,6359,"IHBAS, Dilshad Garden,New Delhi - CPCB",12234735,pm25,µg/m³
29,6359,"IHBAS, Dilshad Garden,New Delhi - CPCB",17790,pm25,µg/m³
44,6924,"Indirapuram, Ghaziabad - UPPCB",19886,pm10,µg/m³
45,6924,"Indirapuram, Ghaziabad - UPPCB",12235778,pm10,µg/m³


In [9]:
test_sensor = int(sensors[sensors["parameter"] == "pm25"]["sensor_id"].iloc[0])
url = f"https://api.openaq.org/v3/sensors/{test_sensor}/measurements/hourly"
params = {
    "datetime_from": "2025-12-01T00:00:00Z",
    "datetime_to": "2025-12-08T00:00:00Z",
    "limit": 1000,
}
r = requests.get(url, headers=headers, params=params)
print("status:", r.status_code)
res = r.json()["results"]
print(len(res), "rows")
print(json.dumps(res[0], indent=2))

status: 200
164 rows
{
  "value": 200.0,
  "flagInfo": {
    "hasFlags": false
  },
  "parameter": {
    "id": 2,
    "name": "pm25",
    "units": "\u00b5g/m\u00b3",
    "displayName": null
  },
  "period": {
    "label": "1 hour",
    "interval": "01:00:00",
    "datetimeFrom": {
      "utc": "2025-11-30T23:30:00Z",
      "local": "2025-12-01T05:00:00+05:30"
    },
    "datetimeTo": {
      "utc": "2025-12-01T00:30:00Z",
      "local": "2025-12-01T06:00:00+05:30"
    }
  },
  "coordinates": null,
  "summary": {
    "min": 199.0,
    "q02": 199.04,
    "q25": 199.5,
    "median": 200.0,
    "q75": 200.5,
    "q98": 200.96,
    "max": 201.0,
    "avg": 200.0,
    "sd": 1.4142135623730951
  },
  "coverage": {
    "expectedCount": 4,
    "expectedInterval": "01:00:00",
    "observedCount": 2,
    "observedInterval": "00:30:00",
    "percentComplete": 50.0,
    "percentCoverage": 50.0,
    "datetimeFrom": {
      "utc": "2025-12-01T00:00:00Z",
      "local": "2025-12-01T05:30:00+05:30"
   